<a href="https://colab.research.google.com/github/alifia07nisa/data-science-2026/blob/main/Pertemuan12_Alifia_Choirunnisa_250401020010.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nama : Alifia Choirunnisa

NIM : 250401020010

Kelas : IF403


In [16]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
 'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']
# Buat 50 transaksi, tiap transaksi berisi 2-5 produk
transaksi = []
for _ in range(50):
 n_item = np.random.randint(2, 6)
 transaksi.append(list(np.random.choice(produk, n_item, replace=False)))
# Suntikkan pola: Roti sering bersama Selai
for i in range(0, 20):
 if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
  transaksi[i].append('Selai')
print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))

Contoh transaksi: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Jumlah transaksi: 50


In [17]:
from mlxtend.preprocessing import TransactionEncoder
te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)
print(df.head())

    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


In [18]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from mlxtend.frequent_patterns import apriori
for ms in [0.05, 0.1, 0.2]:
  freq = apriori(df, min_support=ms, use_colnames=True)
  print(f'min_support={ms}: {len(freq)} itemset ditemukan')
# Gunakan min_support yang menghasilkan jumlah itemset wajar (tidak 0, tidak ratusan)
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)
print(freq_items.head(10))

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Teh, Selai)


In [19]:
from mlxtend.frequent_patterns import association_rules
rules = association_rules(freq_items, metric='confidence',
 min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)
print(rules[['antecedents', 'consequents',
 'support', 'confidence', 'lift']].head(10))


         antecedents consequents  support  confidence      lift
8        (Teh, Keju)     (Telur)     0.12    0.857143  2.380952
13  (Selai, Mentega)      (Kopi)     0.10    0.625000  1.953125
12      (Gula, Roti)     (Selai)     0.10    1.000000  1.923077
7           (Sereal)   (Mentega)     0.14    0.777778  1.851852
9       (Teh, Telur)      (Keju)     0.12    0.600000  1.764706
14     (Selai, Kopi)   (Mentega)     0.10    0.714286  1.700680
10     (Telur, Keju)       (Teh)     0.12    0.750000  1.630435
11     (Gula, Selai)      (Roti)     0.10    0.500000  1.562500
15   (Mentega, Kopi)     (Selai)     0.10    0.714286  1.373626
1             (Roti)     (Selai)     0.22    0.687500  1.322115


Aturan mana yang paling kuat (Lift tertinggi)?

Aturan yang paling kuat yaitu (Teh, Keju) -> (Telur). Nilai Lift 2.38 menunjukkan pelanggan yang membeli Teh dan Keju bersamaan memiliki kemungkinan 2,38 kali lebih besar untuk membeli Telur. Karena Lift>1 maka hubungan ini bersifat positif dan sangat kuat.

Confidence 0.857 menunjukkan dari seluruh transaksi yang memuat Teh dan Keju, 85,7% diantaranya juga memuat Telur.

Support 0.12 menunjukkan kombinasi ketiga item ini muncul sebanyak 12% dari total keseluruhan transaksi.

Apakah masuk akal secara bisnis (mis. Roti -> Selai)?

Ya. Aturan Roti → Selai sangat masuk akal secara bisnis karena kedua item tersebut sering dibeli bersamaan sebagai pelengkap makanan (selai sebagai pelengkap roti).


In [20]:
from sklearn.metrics.pairwise import cosine_similarity
katalog = pd.DataFrame({
 'produk': produk,
 'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy',
 'Dairy','Minuman','Bumbu','Minuman','Dairy']
})
fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)
def rekomendasi_serupa(nama_produk, top_n=3):
 idx = katalog.index[katalog['produk'] == nama_produk][0]
 skor = list(enumerate(sim_matrix[idx]))
 skor = sorted(skor, key=lambda x: x[1], reverse=True)
 skor = [s for s in skor if s[0] != idx][:top_n]
 return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()
print('Mirip dengan Roti:', rekomendasi_serupa('Roti'))

Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


In [21]:
produk_target = 'Roti'
# Dari association rules: cari consequents dari aturan yang antecedent-nya mengandung
produk_target
rules_terkait = rules[rules['antecedents'].apply(
 lambda x: produk_target in x)]
print('Rekomendasi dari Association Rules:')
print(rules_terkait[['consequents', 'lift']].head())
print('Rekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))

Rekomendasi dari Association Rules:
   consequents      lift
12     (Selai)  1.923077
1      (Selai)  1.322115
Rekomendasi dari Content-Based: ['Selai', 'Sereal', 'Susu']


Apakah kedua pendekatan memberi rekomendasi yang konsisten?

Content-Based Filtering merekomendasikan Selai, Sereal, dan Susu.
Association Rules merekomendasikan Selai. Kedua pendekatan merekomendasikan "Selai" sebagai produk utama untuk dibeli bersama atau setelah melihat "Roti" sehingga dapat dianggap kedua pendekatan memberi rekomendasi yang konsisten.

Kapan sebaiknya menggunakan salah satu, atau menggabungkan keduanya (hybrid)?

Association Rules dapat digunakan ketika tersedia banyak data transaksi sehingga pola pembelian pelanggan dapat dianalisis.

Content-Based Filtering dapat digunakan ketika belum memiliki data transaksi (data transaksi masih terbatas) dan ingin memberikan rekomendasi berdasarkan pada karakteristik produk.

Hybrid, kedua metode dapat digabungkan untuk menghasilkan rekomendasi yang lebih akurat dan bervariasi.

Pada pertemuan ke 12 dipelajari Association Rules dengan algoritma Apriori dan Content-Based Filtering yang digunakan untuk menemukan pola pembelian serta memberikan rekomendasi produk.

Association rules yang terkuat, (Teh, Keju) → (Telur) dengan lift 2,38, menunjukkan lebih kuat dibanding pola yang sengaja ditentukan (Roti → Selai) dengan lift 1,32.

Keterbatasan pada praktik kali ini yaitu dataset terbatas yang hanya terdiri dari 50 transaksi sintetis dan Content-Based Filtering hanya menggunakan kategori produk sebagai fitur.